## Phase 1

In [21]:
# ==========================================
# PHASE 1 - DASHBOARD FOUNDATION
# ==========================================

# ---------- Imports ----------
import pandas as pd
import numpy as np

from dash import Dash, html, dcc, Input, Output
import dash_bootstrap_components as dbc

# Plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# Theme Colors
# ==========================================

BACKGROUND = "#0F172A"

CARD = "#1E293B"

TEXT = "#F8FAFC"

PRIMARY = "#3B82F6"

SUCCESS = "#22C55E"

WARNING = "#FACC15"

DANGER = "#EF4444"


SEGMENT_COLORS = {

    "Champions": "#FACC15",

    "Loyal Customers": "#22C55E",

    "Potential Loyalists": "#3B82F6",

    "Regular Customers": "#94A3B8",

    "New Customers": "#06B6D4",

    "At Risk": "#F97316",

    "Hibernating": "#A855F7",

    "Lost Customers": "#EF4444"

}


## Phase 2

In [4]:
# ==========================================
# Segment Summary
# ==========================================

segment_counts = (
    df["Segment"]
    .value_counts()
    .reset_index()
)

segment_counts.columns = [
    "Segment",
    "Customers"
]

segment_counts["Percentage"] = (
    segment_counts["Customers"]
    / segment_counts["Customers"].sum()
) * 100

segment_counts["Percentage"] = (
    segment_counts["Percentage"]
    .round(2)
)

print(segment_counts)


# ==========================================
# Segment Distribution Bar Chart
# ==========================================

bar_fig = px.bar(

    segment_counts,

    x="Segment",

    y="Customers",

    color="Segment",

    color_discrete_map=SEGMENT_COLORS,

    text="Customers"

)

bar_fig.update_traces(

    textposition="outside",

    marker_line_width=1,

    marker_line_color="white",

    hovertemplate=

    "<b>%{x}</b><br>" +

    "Customers : %{y}<extra></extra>"

)

bar_fig.update_layout(

    title="Customer Distribution by Segment",

    title_x=0.5,

    height=520,

    paper_bgcolor=CARD,

    plot_bgcolor=CARD,

    font_color=TEXT,

    showlegend=False,

    margin=dict(

        l=20,

        r=20,

        t=60,

        b=20

    )

)

bar_fig.update_xaxes(

    showgrid=False,

    tickangle=-20

)

bar_fig.update_yaxes(

    gridcolor="#334155"

)

               Segment  Customers  Percentage
0       Lost Customers        889       20.61
1    Regular Customers        850       19.70
2  Potential Loyalists        778       18.03
3          Hibernating        721       16.71
4            Champions        469       10.87
5        New Customers        258        5.98
6      Loyal Customers        254        5.89
7              At Risk         95        2.20


In [5]:
# ==========================================
# Customer Segment Share
# ==========================================

pie_fig = px.pie(

    segment_counts,

    names="Segment",

    values="Customers",

    hole=0.65,

    color="Segment",

    color_discrete_map=SEGMENT_COLORS

)

pie_fig.update_traces(

    textinfo="percent+label",

    textfont_size=13,

    hovertemplate=
    "<b>%{label}</b><br>" +
    "Customers : %{value}<br>" +
    "Percentage : %{percent}<extra></extra>"

)

pie_fig.add_annotation(

    x=0.5,

    y=0.5,

    text=f"<b>{total_customers:,}</b><br>Customers",

    showarrow=False,

    font=dict(

        size=24,

        color="white"

    )

)

pie_fig.update_layout(

    title="Customer Segment Share",

    title_x=0.5,

    paper_bgcolor=CARD,

    plot_bgcolor=CARD,

    font_color=TEXT,

    height=520,

    legend=dict(

        orientation="h",

        y=-0.15,

        x=0.5,

        xanchor="center"

    )

)

# Phase 3

In [8]:
# ==========================================
# Bubble Chart
# ==========================================

bubble_fig = px.scatter(

    df,

    x="Frequency",

    y="Monetary",

    size="Monetary",

    color="Segment",

    color_discrete_map=SEGMENT_COLORS,

    hover_name="Customer ID",

    hover_data={

        "Recency": True,

        "Frequency": True,

        "Monetary": ":,.0f",

        "Score": True,

        "Segment": False

    },

    size_max=60

)

bubble_fig.update_traces(

    marker=dict(

        opacity=0.75,

        line=dict(

            width=1,

            color="white"

        )

    )

)

bubble_fig.update_layout(

    title="Customer Behaviour Analysis",

    title_x=0.5,

    paper_bgcolor=CARD,

    plot_bgcolor=CARD,

    font_color=TEXT,

    height=650,

    xaxis_title="Purchase Frequency",

    yaxis_title="Monetary Value",

    legend_title="Customer Segment",

    transition_duration=700

)

bubble_fig.update_xaxes(

    showgrid=True,

    gridcolor="#334155",

    zeroline=False

)

bubble_fig.update_yaxes(

    showgrid=True,

    gridcolor="#334155",

    zeroline=False
)

In [24]:

# ---------- Load Dataset ----------
df = pd.read_csv("datasets/rfm_main.csv")

# ---------- Theme Colors ----------
BACKGROUND = "#0F172A"   # Page background
CARD = "#1E293B"         # Card background
TEXT = "#F8FAFC"         # Main text

# ---------- KPI Calculations ----------
total_customers = df["Customer ID"].nunique()
total_revenue = df["Monetary"].sum()
avg_monetary = round(df["Monetary"].mean(), 0)
champions = (df["Segment"] == "Champions").sum()

# ---------- KPI Card Function ----------
def create_kpi_card(title, value, color):
    return dbc.Card(
        dbc.CardBody([
            html.H6(
                title,
                style={
                    "color": "#CBD5E1",
                    "fontWeight": "600",
                    "marginBottom": "10px"
                }
            ),
            html.H2(
                value,
                style={
                    "color": color,
                    "fontWeight": "bold",
                    "margin": 0
                }
            )
        ]),
        style={
            "backgroundColor": CARD,
            "border": "none",
            "borderRadius": "18px",
            "padding": "10px",
            "boxShadow": "0px 4px 20px rgba(0,0,0,0.25)"
        }
    )

# ---------- Initialize App ----------
app = Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP]
)

server = app.server

# ---------- Layout ----------
app.layout = dbc.Container([

    # Top spacing
    html.Br(),

    # Title
    html.H1(
        "Retail Customer Analytics Dashboard",
        style={
            "textAlign": "center",
            "color": TEXT,
            "fontWeight": "bold",
            "fontSize": "42px",
            "marginBottom": "5px"
        }
    ),

    # Subtitle
    html.P(
        "RFM Based Customer Segmentation",
        style={
            "textAlign": "center",
            "color": "#94A3B8",
            "fontSize": "18px",
            "marginBottom": "25px"
        }
    ),

    html.Hr(style={"borderColor": "#334155"}),

    # KPI Cards Row
    dbc.Row([

        dbc.Col(
            create_kpi_card(
                "Customers",
                f"{total_customers:,}",
                "#38BDF8"
            ),
            width=3
        ),

        dbc.Col(
            create_kpi_card(
                "Revenue",
                f"₹ {total_revenue:,.0f}",
                "#22C55E"
            ),
            width=3
        ),

        dbc.Col(
            create_kpi_card(
                "Average Monetary",
                f"₹ {avg_monetary:,.0f}",
                "#FACC15"
            ),
            width=3
        ),

        dbc.Col(
            create_kpi_card(
                "Champions",
                f"{champions:,}",
                "#FB7185"
            ),
            width=3
        )

    ], className="mb-4"),
    html.Br(),

dbc.Row([

    dbc.Col(

        dcc.Dropdown(

            id="segment_filter",

            options=[

                {"label": "All Segments", "value": "All"}

            ] + [

                {

                    "label": seg,

                    "value": seg

                }

                for seg in sorted(df["Segment"].unique())

            ],

            value="All",

            clearable=False,

            style={

                "color": "black"

            }

        ),

        width=4

    ),

    dbc.Col(

        dcc.Dropdown(

            id="rscore_filter",

            options=[

                {"label":"All R Scores","value":"All"},

                {"label":"1","value":1},

                {"label":"2","value":2},

                {"label":"3","value":3},

                {"label":"4","value":4},

                {"label":"5","value":5}

            ],

            value="All",

            clearable=False,

            style={

                "color":"black"

            }

        ),

        width=4

    ),

    dbc.Col(

        html.Div(

            "Interactive Filters",

            style={

                "color":TEXT,

                "fontSize":"20px",

                "fontWeight":"bold",

                "textAlign":"center",

                "paddingTop":"8px"

            }

        ),

        width=4

    )

]),

    # Placeholder section for future charts
    html.Br(),

dbc.Row(

    [

        dbc.Col(

            dbc.Card(

                dbc.CardBody(

                    dcc.Graph(

                    id="segment_chart",

                    figure=bar_fig
                    )
                ),

                style={

                    "backgroundColor": CARD,

                    "border": "none",

                    "borderRadius": "20px"

                }

            ),

            width=7

        ),

        dbc.Col(

            dbc.Card(

                dbc.CardBody(

                    dcc.Graph(

                        figure=pie_fig,

                        config={

                            "displayModeBar": False

                        }

                    )

                ),

                style={

                    "backgroundColor": CARD,

                    "border": "none",

                    "borderRadius": "20px"

                }

            ),

            width=5

        )

    ]

),
html.Br(),

dbc.Row(

    [

        dbc.Col(

            dbc.Card(

                dbc.CardBody(

                    dcc.Graph(

                        figure=bubble_fig,

                        config={

                            "displayModeBar": False

                        }

                    )

                ),

                style={

                    "backgroundColor": CARD,

                    "border": "none",

                    "borderRadius": "20px"

                }

            ),

            width=12

        )

    ]

),
html.Br(),

dbc.Row(

    [

        dbc.Col(

            dbc.Card(

                dbc.CardBody(

                    dcc.Graph(

                        figure=revenue_fig,

                        config={

                            "displayModeBar": False

                        }

                    )

                ),

                style={

                    "backgroundColor": CARD,

                    "border": "none",

                    "borderRadius": "20px"

                }

            ),

            width=12

        )

    ]

),
html.Br(),

dbc.Row(

    [

        dbc.Col(

            dbc.Card(

                dbc.CardBody(

                    dcc.Graph(

                        figure=box_fig,

                        config={

                            "displayModeBar": False

                        }

                    )

                ),

                style={

                    "backgroundColor": CARD,

                    "border": "none",

                    "borderRadius": "20px"

                }

            ),

            width=12

        )

    ]

),
html.Br(),

dbc.Row(

    [

        dbc.Col(

            dbc.Card(

                dbc.CardBody(

                    dcc.Graph(

                        figure=top_customer_fig,

                        config={

                            "displayModeBar": False

                        }

                    )

                ),

                style={

                    "backgroundColor": CARD,

                    "border": "none",

                    "borderRadius": "20px"

                }

            ),

            width=12

        )

    ]

),
html.Br(),

dbc.Row(

    [

        dbc.Col(

            dbc.Card(

                dbc.CardBody(

                    dcc.Graph(

                        figure=heatmap_fig,

                        config={

                            "displayModeBar": False

                        }

                    )

                ),

                style={

                    "backgroundColor": CARD,

                    "border": "none",

                    "borderRadius": "20px"

                }

            ),

            width=12

        )

    ]

),

], fluid=True, style={
    "backgroundColor": BACKGROUND,
    "minHeight": "100vh",
    "padding": "30px"
})

# ---------- Run Server ----------
if __name__ == "__main__":
    app.run(debug=True)

In [10]:
# ==========================================
# Revenue by Segment
# ==========================================

segment_revenue = (

    df.groupby("Segment")["Monetary"]

    .sum()

    .reset_index()

    .sort_values(

        by="Monetary",

        ascending=True

    )

)

segment_revenue["Monetary"] = segment_revenue["Monetary"].round(2)

In [11]:
# ==========================================
# Revenue Chart
# ==========================================

revenue_fig = px.bar(

    segment_revenue,

    x="Monetary",

    y="Segment",

    orientation="h",

    color="Segment",

    color_discrete_map=SEGMENT_COLORS,

    text="Monetary"

)

revenue_fig.update_traces(

    texttemplate="₹ %{x:,.0f}",

    textposition="outside",

    marker_line_color="white",

    marker_line_width=1.2,

    hovertemplate=

    "<b>%{y}</b><br>" +

    "Revenue : ₹ %{x:,.0f}<extra></extra>"

)

revenue_fig.update_layout(

    title="Revenue Contribution by Customer Segment",

    title_x=0.5,

    paper_bgcolor=CARD,

    plot_bgcolor=CARD,

    font_color=TEXT,

    showlegend=False,

    height=600,

    margin=dict(

        l=30,

        r=30,

        t=60,

        b=20

    )

)

revenue_fig.update_xaxes(

    gridcolor="#334155"

)

revenue_fig.update_yaxes(

    showgrid=False
)

In [13]:
# ==========================================
# Monetary Distribution by Segment
# ==========================================

box_fig = px.box(

    df,

    x="Segment",

    y="Monetary",

    color="Segment",

    color_discrete_map=SEGMENT_COLORS,

    points="outliers"

)

box_fig.update_traces(

    hovertemplate=

    "<b>%{x}</b><br>" +

    "Monetary : ₹ %{y:,.0f}<extra></extra>"

)

box_fig.update_layout(

    title="Monetary Distribution Across Customer Segments",

    title_x=0.5,

    paper_bgcolor=CARD,

    plot_bgcolor=CARD,

    font_color=TEXT,

    height=650,

    showlegend=False,

    xaxis_title="Customer Segment",

    yaxis_title="Monetary Value"

)

box_fig.update_xaxes(

    showgrid=False,

    tickangle=-20

)

box_fig.update_yaxes(

    gridcolor="#334155"

)

In [15]:
# ==========================================
# Top 20 Customers
# ==========================================

top_customers = (

    df

    .sort_values(

        by="Monetary",

        ascending=False

    )

    .head(20)

)

In [16]:
# ==========================================
# Top Customers Chart
# ==========================================

top_customer_fig = px.bar(

    top_customers,

    x="Monetary",

    y=top_customers["Customer ID"].astype(str),

    orientation="h",

    color="Segment",

    color_discrete_map=SEGMENT_COLORS,

    text="Monetary"

)

top_customer_fig.update_traces(

    texttemplate="₹ %{x:,.0f}",

    textposition="outside",

    hovertemplate=

    "<b>Customer %{y}</b><br>" +

    "Revenue : ₹ %{x:,.0f}<br>" +

    "Frequency : %{customdata[0]}<br>" +

    "Recency : %{customdata[1]} Days<extra></extra>",

    customdata=top_customers[["Frequency","Recency"]]

)

top_customer_fig.update_layout(

    title="Top 20 Customers by Revenue",

    title_x=0.5,

    paper_bgcolor=CARD,

    plot_bgcolor=CARD,

    font_color=TEXT,

    height=700,

    showlegend=True,

    xaxis_title="Revenue",

    yaxis_title="Customer ID"

)

top_customer_fig.update_xaxes(

    gridcolor="#334155"

)

top_customer_fig.update_yaxes(

    autorange="reversed"

)

In [18]:
# ==========================================
# Correlation Matrix
# ==========================================

corr = df[

    [

        "Recency",

        "Frequency",

        "Monetary"

    ]

].corr()

print(corr)

            Recency  Frequency  Monetary
Recency    1.000000  -0.248928 -0.121209
Frequency -0.248928   1.000000  0.656273
Monetary  -0.121209   0.656273  1.000000


In [19]:
# ==========================================
# Correlation Heatmap
# ==========================================

heatmap_fig = px.imshow(

    corr,

    text_auto=".2f",

    color_continuous_scale="RdBu_r",

    aspect="auto"

)

heatmap_fig.update_layout(

    title="Correlation between RFM Features",

    title_x=0.5,

    paper_bgcolor=CARD,

    plot_bgcolor=CARD,

    font_color=TEXT,

    height=550,

    coloraxis_colorbar=dict(

        title="Correlation"

    )

)

In [23]:
# Bar chart call back
@app.callback(

    Output("segment_chart","figure"),

    [

        Input("segment_filter","value"),

        Input("rscore_filter","value")

    ]

)

def update_bar(segment,rscore):

    filtered = df.copy()

    if segment != "All":

        filtered = filtered[

            filtered["Segment"] == segment

        ]

    if rscore != "All":

        filtered = filtered[

            filtered["R_score"] == rscore

        ]

    segment_counts = (

        filtered["Segment"]

        .value_counts()

        .reset_index()

    )

    segment_counts.columns=[

        "Segment",

        "Customers"

    ]

    fig = px.bar(

        segment_counts,

        x="Segment",

        y="Customers",

        color="Segment",

        color_discrete_map=SEGMENT_COLORS

    )

    fig.update_layout(

        paper_bgcolor=CARD,

        plot_bgcolor=CARD,

        font_color=TEXT,

        showlegend=False

    )

    return fig